In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import xarray as xr
import numpy as np
import os
from glob import glob
from mpl_toolkits.basemap import Basemap
from numpy import meshgrid
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
import cartopy.feature as cfeature
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter, LatitudeLocator
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, TwoSlopeNorm
import pandas as pd
import math
from datetime import datetime
import datetime as dt
from ridgeplot import ridgeplot
import joypy
import seaborn as sns
from matplotlib import cm
import climpred
from xclim import sdba
from climpred.options import OPTIONS
import json
from sklearn.metrics import roc_curve, auc, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from matplotlib.lines import Line2D  # For custom legend entries
import warnings
from sklearn.exceptions import UndefinedMetricWarning
import matplotlib.gridspec as gridspec
import hydroeval as he
import re
from matplotlib.ticker import FormatStrFormatter

from function import preprocessUtils as putils
from function import masks
from function import verifications
from function import funs as f
from function import conf
from function import loadbias
from function import dataLoad


warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

global dim_order, region_name, test_year, leads_
dim_order = conf.dim_order
region_name = 'CONUS'
test_year = 2019
leads_ = [6,13,20,27]

dir = '/glade/work/klesinger/FD_RZSM_deep_learning'
assert test_year == 2019, 'This is only the script for when the testing years are 2018-2019. Test year must = 2019.'

mask, mask_anom = masks.load_mask_vals(region_name)


/glade/work/klesinger/conda-envs/tf212gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
obs_source = 'GLEAM'

if obs_source == 'GLEAM':
    soil_dir = conf.gleam_data
elif obs_source == 'ERA5':
    soil_dir = conf.era_data

In [3]:
'''Testing and validation dates only for the year 2019'''

test_start = '2018-01-01'
test_end = '2019-12-31'
val_start = '2016-01-01'
val_end = '2017-12-31'
train_start  = '2000-01-01'


'''Test subsets of obs, ecmwf raw, gefsv12 raw '''
global obs_anomaly_SubX_format, baseline_gefs, baseline_ecmwf, var_OUT, template_testing_only
obs_anomaly_SubX_format, baseline_gefs, baseline_ecmwf, var_OUT, template_testing_only = verifications.open_obs_and_baseline_files_multiple_leads(region_name, leads_, test_start, test_end, mask_anom, soil_dir)

#Open the gleam percentile/anom files
#Open the gleam percentile/anom files
init_dates,dt_dates,only_testing_dates = dataLoad.return_init_and_testing_dates(region_name,test_start,test_end)



Loading soil data all the baseline files for observations, GEFSv12, and ECMWF for region CONUS
Loading /glade/derecho/scratch/klesinger/FD_RZSM_deep_learning/Data/reanalysis/GLEAM observations
Loading GEFS data
Loading ECMWF data


In [4]:
global obs_original,obs_raw,gef_BC,ecm_BC
obs_original,obs_raw = dataLoad.load_rzsm_observations(soil_dir, region_name)

gef_BC, ecm_BC = loadbias.load_additive_bias_anomaly([6,13,20,27],region_name,obs_source)


In [5]:
gef_BC = gef_BC.rename({'init':'S','member':'M','lead':'L','lat':'Y','lon':'X'})
ecm_BC = ecm_BC.rename({'init':'S','member':'M','lead':'L','lat':'Y','lon':'X'})

In [ ]:
def common_UNET_experiments(correct_experiments):
    only_RZSM = [j for j in correct_experiments if 'RZSM' in j] 
    only_ensemble= [j for j in only_RZSM if 'final' not in j]
    only_ensemble = [j for j in only_ensemble if 'Residual' not in j]
    only_2019 = [j for j in only_ensemble if '2012' not in j]
    only_2019 = [j for j in only_ensemble if 'ERA5' not in j]
    return(only_2019)

def common_UNET_no_regular_experiments(correct_experiments):
    only_RZSM = [j for j in correct_experiments if 'RZSM' in j] 
    only_ensemble= [j for j in only_RZSM if 'final' not in j]
    only_ensemble = [j for j in only_ensemble if 'Residual' not in j]
    only_ensemble = [j for j in only_ensemble if 'regular' not in j]
    only_2019 = [j for j in only_ensemble if '2012' not in j]
    return(only_2019)

In [163]:
def KGE(obs_array, forecast_array):
    out_array = np.empty(shape=(obs_array.shape[1], obs_array.shape[2]))
    
    for Y in range(obs_array.shape[1]):
        for X in range(obs_array.shape[2]):
            obs = obs_array[:, Y, X]
            pred = forecast_array[:, Y, X]
            
            if np.any(np.isnan(obs)) or np.any(np.isnan(pred)):
                out_array[Y, X] = np.nan
            else:
                kge, r, alpha, beta = he.evaluator(he.kge, pred, obs)
                out_array[Y, X] = kge
    return out_array

In [8]:
def take_mean_and_reduce_dimension(file):
    return(file[putils.xarray_varname(file)].mean(dim='M').isel(L=0).values)

In [ ]:
def spatial_KGE_ensemble_mean_multiple_experiments(week_lead, region_name, test_start, test_end, ex1, ex2):

    # week_lead = 3
    # ex1='EX29'
    # ex2 = 'EX24'
    
    save_dir = f'Outputs/KGE_spatial_plots/{region_name}'
    os.system(f'mkdir -p {save_dir}')
    
    plot_dict = {}

    day_num = (week_lead*7) -1

    print('Loading observation and baseline anomaly files')
    obs, gefs, ecmwf = obs_anomaly_SubX_format.sel(L=day_num).expand_dims({'L':1}).transpose(*dim_order), gef_BC.sel(L=day_num).expand_dims({'L':1}).transpose(*dim_order), ecm_BC.sel(L=day_num).expand_dims({'L':1}).transpose(*dim_order)

    obs_arr = take_mean_and_reduce_dimension(obs) #(104, 48, 96)
    gefs_arr = take_mean_and_reduce_dimension(gefs) #(104, 48, 96)
    ecmwf_arr = take_mean_and_reduce_dimension(ecmwf) #(104, 48, 96)

    gefs_cal = KGE(obs_arr, gefs_arr)
    ecmwf_cal = KGE(obs_arr, ecmwf_arr)

    gefs_out = obs.mean(dim='M').isel(S=0).isel(L=0).copy(deep=True)
    gefs_out[putils.xarray_varname(gefs_out)][:,:] = gefs_cal

    ecmwf_out = obs.mean(dim='M').isel(S=0).isel(L=0).copy(deep=True)
    ecmwf_out[putils.xarray_varname(ecmwf_out)][:,:] = ecmwf_cal
    
    plot_dict.update({'ECMWF-BC':ecmwf_out})
    plot_dict.update({'GEFSv12-BC':gefs_out})
    
    ####################################################################################################################################


    unet_files = sorted(glob(f'predictions/{region_name}/Wk{week_lead}_testing/*{ex1}*'))

    ec_unet = f'predictions/CONUS/Wk{week_lead}_testing/Wk{week_lead}_testing_{ex1}_ECMWF_regular_RZSM.npy'
    gef_unet = f'predictions/CONUS/Wk{week_lead}_testing/Wk{week_lead}_testing_{ex1}_regular_RZSM.npy'
    
    
    print('Working on UNET experiments')
    unet_files = [ec_unet,gef_unet]
    #Now loop through and open file, convert to anomaly and compute ACC score
    for i in unet_files:
        # i=unet_files[0]     

        new_source = 'GEFSv12'

        test_name = i.split('testing_')[-1].split('.npy')[0]
        
        # break
        add_to_file = verifications.load_UNET_files(gefs=gefs, file=i, region_name=region_name, day_num=day_num,new_source=new_source,test_year=test_year)
        unet_arr = take_mean_and_reduce_dimension(add_to_file) #(104, 48, 96)
        unet_cal = KGE(obs_arr, unet_arr)
        unet_out = obs.mean(dim='M').isel(S=0).isel(L=0).copy(deep=True)
        unet_out[putils.xarray_varname(unet_out)][:,:] = unet_cal
        
        plot_dict.update({test_name:unet_out})

    
    #Get global max and min
    
    global_max, global_min = verifications.global_max_min(plot_dict,'var')
    v = np.linspace(-3, global_max, 20, endpoint=True)
    v = [i for i in v if i <0] + [0] + [i for i in v if i >0] 
    norm = TwoSlopeNorm(0, vmin=v[0], vmax=v[-1])
    
    cmap = 'bwr'
    
    fig, axs = plt.subplots(
        nrows = 3, ncols= 2, subplot_kw={'projection': ccrs.PlateCarree()}, figsize=(10, 8), dpi=300)
    
    axs = axs.flatten()
    
    lon = mask.X.values
    lat = mask.Y.values

    # plot_dictionary = {} #This will save the statistics for each model to see where the ACC skill is greater than a specific threshold
    # total_grid_cells = np.count_nonzero(mask_anom)
    
    axs_start = 0
    for model in plot_dict.keys():
        # plot_dictionary[model] = {}
        # break
        data = plot_dict[model][putils.xarray_varname(plot_dict[model])].values

        # for threshold in [0.4,0.5,0.6,0.7,0.8,0.9]:
        #     abv_thresh = np.where(data>threshold,1,0)
        #     plot_dictionary[model][threshold] = np.count_nonzero(abv_thresh) / total_grid_cells
            
    
        map = Basemap(projection='cyl', llcrnrlat=lat[-1] - 1.5, urcrnrlat=lat[0],
                      llcrnrlon=(lon[0] - 360 - 6), urcrnrlon=(lon[-1] - 360 + 20), resolution='l')
        
        x, y = map(*np.meshgrid(lon, lat))
        # Adjust the text coordinates based on the actual data coordinates

        im = axs[axs_start].contourf(x, y, data, levels=v, extend='both',
                              transform=ccrs.PlateCarree(), cmap=cmap,norm=norm)
        
        gl = axs[axs_start].gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                                   linewidth=0.7, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        gl.left_labels = True
        gl.xformatter = LongitudeFormatter()
        gl.yformatter = LatitudeFormatter()
        gl.xlabel_style = {'size': 7}  # Adjust the font size as per your preference
        gl.ylabel_style = {'size': 7}  # Adjust the font size as per your preference
        
        axs[axs_start].coastlines()
        axs[axs_start].set_aspect('equal')  # this makes the plots better

        FSIZE = 12
        if model == f'{ex1}_ECMWF_regular_RZSM':
            # axs[axs_start].set_title(f'EXP{ex1[2:]}_ECMWF',fontsize=11)
            axs[axs_start].set_title(f'(C) UNet_E',fontsize=FSIZE)
            axs[axs_start].title.pad = 10
        elif model == f'{ex1}_regular_RZSM':
            axs[axs_start].set_title(f'(D) UNet_G', fontsize=FSIZE)
            # axs[axs_start].set_title(f'EXP{ex1[2:]}_GEFSv12', fontsize=11)
            axs[axs_start].title.pad = 10
        elif model == f'{ex2}_regular_RZSM':
            axs[axs_start].set_title(f'(E) EXP24_OBS', fontsize=FSIZE)
            # axs[axs_start].set_title(f'EXP{ex2[2:]}_OBS', fontsize=11)
        else:
            if model == 'ECMWF':
                axs[axs_start].set_title(f'(A) {model}',fontsize=FSIZE)
            else:
                axs[axs_start].set_title(f'(B) {model}', fontsize=FSIZE)
            
        axs_start+=1

                            #(left, bottom, width, height)
    cbar_ax = fig.add_axes([0.13, 0.05, .76, .03])
    
    # Draw the colorbar
    cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
    # plt.suptitle(f'ACC on testing Dataset', fontsize=30)
    # plt.tight_layout()

    # plot_dictionary['total_grid_cells'] = total_grid_cells
    
    plt.savefig(f'{save_dir}/Wk{week_lead}_KGE_multiple_experiments.png')

    return(f'Completed KGE plots lead {week_lead}.')
    

In [ ]:
'''We only ran EMOS, XGBoost on CONUS'''
for week_lead in [3]:
    if region_name == 'CONUS':
        plot_dictionary = spatial_ACC_climpred_with_XGBOOST_multiple_experiments(week_lead, region_name, test_start, test_end, ex1='EX29', ex2='EX24')
        del plot_dictionary['EX24_regular_RZSM_2012']
        p = plot_dictionary.copy()
        save_ACC_threshold_values(plot_dict = p)


In [ ]:
'''KGE plots'''
for week_lead in [1,2,3,4,5]:
    if region_name == 'CONUS':
        spatial_KGE_ensemble_mean_multiple_experiments(week_lead, region_name, test_start, test_end, ex1='EX29', ex2='EX24')




In [208]:
def spatial_KGE_ensemble_mean_multiple_experiments_seasonal_all_in_one(week_lead, region_name, test_start, test_end, ex1):
    import calendar
    import matplotlib.pyplot as plt
    from matplotlib.colors import TwoSlopeNorm
    import cartopy.crs as ccrs
    from mpl_toolkits.basemap import Basemap
    import os
    import numpy as np

    save_dir = f'Outputs/KGE_spatial_plots/{region_name}/seasonal'
    os.makedirs(save_dir, exist_ok=True)

    day_num = (week_lead * 7) - 1
    print(f'Loading observation and baseline anomaly files for day {day_num}...')

    obs = obs_anomaly_SubX_format.sel(L=day_num).expand_dims({'L': 1}).transpose(*dim_order)
    gefs = gef_BC.sel(L=day_num).expand_dims({'L': 1}).transpose(*dim_order)
    ecmwf = ecm_BC.sel(L=day_num).expand_dims({'L': 1}).transpose(*dim_order)

    total_grid_cells = np.count_nonzero(~np.isnan(obs['var'][0,0,0,:,:].values))
    
    seasons = {
        "DJF": [12, 1, 2],
        "MAM": [3, 4, 5],
        "JJA": [6, 7, 8],
        "SON": [9, 10, 11]
    }

    # One dict: model -> season -> data
    model_seasonal_dict = {}

    def compute_seasonal_KGE(obs, pred):
        result = {}
        for season, months in seasons.items():
            obs_season = obs.sel(S=obs['S.month'].isin(months))
            pred_season = pred.sel(S=pred['S.month'].isin(months))
            obs_arr = take_mean_and_reduce_dimension(obs_season)
            pred_arr = take_mean_and_reduce_dimension(pred_season)
            kge_score = KGE(obs_arr, pred_arr)
            kge_out = obs.mean(dim='M').isel(S=0).isel(L=0).copy(deep=True)
            kge_out[putils.xarray_varname(kge_out)][:, :] = kge_score
            result[season] = kge_out
        return result

    print('Computing seasonal KGE for baseline models...')
    model_seasonal_dict['GEFSv12-BC'] = compute_seasonal_KGE(obs, gefs)
    model_seasonal_dict['ECMWF-BC'] = compute_seasonal_KGE(obs, ecmwf)

    print('Computing seasonal KGE for UNET experiments...')
    gef_unet = f'predictions/{region_name}/Wk{week_lead}_testing/Wk{week_lead}_testing_{ex1}_ECMWF_regular_RZSM.npy'
    ec_unet = f'predictions/{region_name}/Wk{week_lead}_testing/Wk{week_lead}_testing_{ex1}_RZSM.npy'
    
    for idx,i in enumerate([ec_unet, gef_unet]):
        new_source = 'GEFSv12'
        test_name = i.split('testing_')[-1].split('.npy')[0]
        if idx == 0:
            test_name = 'DL-DM_E'
        else:
            test_name = 'DL-DM_G'
        
        add_to_file = verifications.load_UNET_files(
            gefs=gefs, file=i, region_name=region_name,
            day_num=day_num, new_source=new_source, test_year=test_year
        )
        model_seasonal_dict[test_name] = compute_seasonal_KGE(obs, add_to_file)

    print('Building unified seasonal KGE plot...')
    lon = mask.X.values
    lat = mask.Y.values
    model_names = list(model_seasonal_dict.keys())
    season_names = list(seasons.keys())

    # Compute global vmin/vmax across all models and seasons
    combined_dict = {
        season: {model: model_seasonal_dict[model][season] for model in model_names}
        for season in season_names
    }
    # global_max, global_min = verifications.global_max_min(combined_dict, 'var')
    v = np.linspace(-3, 1, 20, endpoint=True)
    v = [i for i in v if i < 0] + [0] + [i for i in v if i > 0]
    norm = TwoSlopeNorm(0, vmin=v[0], vmax=v[-1])
    cmap = 'bwr'

    nrows = len(model_names)
    ncols = len(season_names)

    fig, axs = plt.subplots(
        nrows=nrows, ncols=ncols,
        subplot_kw={'projection': ccrs.PlateCarree()},
        figsize=(4.5 * ncols, 3 * nrows), dpi=300
    )

    if nrows == 1:
        axs = np.expand_dims(axs, 0)  # handle single model
    if ncols == 1:
        axs = np.expand_dims(axs, 1)  # handle single season

    labels = ['(A)', '(B)', '(C)', '(D)']
    for i, model in enumerate(model_names):
        for j, season in enumerate(season_names):
            ax = axs[i][j]
            kge_data = model_seasonal_dict[model][season]
            data = kge_data[putils.xarray_varname(kge_data)].values
            pos = np.round(np.count_nonzero(data>=0)/total_grid_cells,3)
            m = Basemap(
                projection='cyl',
                llcrnrlat=lat[-1] - 1.5, urcrnrlat=lat[0],
                llcrnrlon=(lon[0] - 360 - 6), urcrnrlon=(lon[-1] - 360 + 20), resolution='l',
                ax=ax
            )
            x, y = m(*np.meshgrid(lon, lat))
            im = ax.contourf(x, y, data, levels=v, extend='both',
                             transform=ccrs.PlateCarree(), cmap=cmap, norm=norm)
            ax.coastlines()
            if i == 0:
                ax.set_title(season, fontsize=16)
            if j == 0:
                ax.text(-0.05, 0.5, f'{labels[i]} {model}', va='center', ha='right', fontsize=14, transform=ax.transAxes, rotation=90)
            ax.text(
                0.98, 0.02,          # X, Y coordinates (bottom right in Axes coords)
                f"Percentage above 0: {pos}",    # Text to display
                transform=ax.transAxes,
                fontsize=8,
                ha='right',          # Horizontal alignment (to right edge)
                va='bottom',         # Vertical alignment (to bottom)
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.6)
            )

    # Add colorbar at the bottom
    cbar_ax = fig.add_axes([0.2, 0.08, 0.6, 0.02])
    cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
    cbar.set_label('KGE')

    plt.suptitle(f'Seasonal KGE (Lead {week_lead} weeks)', fontsize=14)
    plt.tight_layout(rect=[0, 0.1, 1, 0.95])
    plt.savefig(f'{save_dir}/Wk{week_lead}_KGE_all_seasons_combined.png')
    plt.close()

    return f'✅ Combined seasonal KGE plot saved for lead {week_lead}.'


In [209]:
spatial_KGE_ensemble_mean_multiple_experiments_seasonal_all_in_one(week_lead, region_name, test_start, test_end, ex1='EX29')

Loading observation and baseline anomaly files for day 20...
Computing seasonal KGE for baseline models...
Computing seasonal KGE for UNET experiments...
Loading UNET testing predictions and reversing the min-max scaling to be back to anomalies
Loading UNET testing predictions and reversing the min-max scaling to be back to anomalies
Building unified seasonal KGE plot...


/glade/derecho/scratch/klesinger/tmp/ipykernel_104726/2067421826.py:135: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.1, 1, 0.95])


'✅ Combined seasonal KGE plot saved for lead 3.'